# **Symmetry Reduction in the Traveling Salesman Problem: ILP Formulation**

This notebook presents an Integer Linear Programming formulation for the traveling salesman problem using the Miller-Tucker-Zemlin subtour elimination constraints and investigates the role of dihedral symmetry. We demonstrate how the rotational and reflective equivalence of tours generates redundant optimal solutions and how a single arc-fixing constraint can eliminate this redundancy while preserving optimality.

---

## *Mathematical Background*

### **Traveling Salesman Problem as Integer Linear Program**

Given a complete directed graph $D = (V, A)$ on $n = |V|$ vertices with arc weights $w_{ij} > 0$, the traveling salesman problem seeks a Hamiltonian cycle — a closed tour visiting every vertex exactly once — of minimum total weight.<sup>[[1]](#ref1)</sup>

We introduce binary decision variables $x_{ij} \in \{0,1\}$ for each arc $(i,j) \in A$ with $i \neq j$, where $x_{ij} = 1$ indicates that the tour traverses the arc from $i$ to $j$. The objective function minimizes the total tour cost:

\begin{equation*}
\min \sum_{i \in V} \sum_{j \in V,\, j \neq i} w_{ij} \cdot x_{ij}
\end{equation*}

Two families of degree constraints ensure that every vertex has exactly one successor and one predecessor in the tour:

\begin{equation*}
\sum_{j \neq i} x_{ij} = 1, \quad \forall i \in V
\qquad
\sum_{i \neq j} x_{ij} = 1, \quad \forall j \in V
\end{equation*}

The degree constraints alone do not guarantee a single connected tour; they also permit solutions consisting of multiple disjoint subtours. To eliminate subtours, we use the **Miller-Tucker-Zemlin (MTZ)** formulation.<sup>[[1]](#ref1), [[2]](#ref2)</sup> We introduce continuous variables $u_i \geq 0$ for each non-depot vertex $i \in V \setminus \{1\}$, representing the position of vertex $i$ in the tour. The MTZ constraints are:

\begin{equation*}
u_i - u_j + n \cdot x_{ij} \leq n - 1, \quad \forall i,j \in V \setminus \{1\},\; i \neq j
\end{equation*}

If $x_{ij} = 1$, the constraint forces $u_j \geq u_i + 1$, imposing a strict ordering on the visit sequence and making it impossible to close a subtour that does not include vertex $1$. Vertex $1$ serves as the depot and is implicitly placed at position $0$ in the ordering.

---

### **Graph Symmetries**

The traveling salesman problem exhibits a rich symmetry structure that generates multiple equivalent representations of every tour.<sup>[[2]](#ref2), [[3]](#ref3)</sup>

A Hamiltonian cycle on $n$ vertices can be traversed starting from any of the $n$ vertices, and in either of two directions. These two transformations generate the **dihedral group** $D_n$, which acts on the set of all tours:

- **Rotational symmetry** $\mathbb{Z}_n$: Cyclically shifting the starting vertex of a tour leaves the set of traversed arcs unchanged in terms of cost. For instance, the tours $1 \to 2 \to 3 \to 4 \to 5 \to 1$ and $2 \to 3 \to 4 \to 5 \to 1 \to 2$ are representations of the same cycle. This generates $n$ equivalent solutions per tour.

- **Reflection symmetry** $\mathbb{Z}_2$: Reversing the direction of traversal yields a tour of identical cost when edge weights are symmetric (i.e., $w_{ij} = w_{ji}$). The tours $1 \to 2 \to 3 \to 4 \to 5 \to 1$ and $1 \to 5 \to 4 \to 3 \to 2 \to 1$ traverse the same edges in opposite directions. This doubles the number of equivalent representations.

Together, these two symmetries form the dihedral group $D_n$ of order $|D_n| = 2n$. Every essentially distinct tour belongs to an orbit of size $2n$ under $D_n$, meaning that each unique tour appears $2n$ times in a full enumeration without symmetry breaking. The number of essentially distinct tours on $n$ vertices is therefore:

\begin{equation*}
\frac{(n-1)!}{2}
\end{equation*}

For $n = 5$, this gives $4!/2 = 12$ essentially distinct tours, while the total number of directed Hamiltonian cycles is $12 \times |D_5| = 12 \times 10 = 120$.

When all edge weights are equal, an additional symmetry arises: any permutation of the vertex labels yields a tour of the same cost, since no vertex or edge is distinguishable. The full symmetry group in this case is the symmetric group $S_n$, which contains $D_n$ as a subgroup. For $n = 5$, $|S_5| = 120$, which equals the total number of directed Hamiltonian cycles: all $120$ tours form a single orbit under $S_5$.

---

### **Symmetry Breaking**

To eliminate the $|D_n| = 2n$ equivalent representations of each tour generated by rotational and reflective symmetry, we impose a single arc-fixing constraint.<sup>[[2]](#ref2), [[3]](#ref3)</sup>

We fix the first arc of the tour by requiring that vertex $1$ is always followed immediately by vertex $2$:

\begin{equation*}
x_{12} = 1
\end{equation*}

This constraint simultaneously breaks both components of $D_n$:

- It breaks **rotational symmetry** by fixing the starting vertex and its successor, so no cyclic shift of the tour can satisfy the constraint unless it already begins with the arc $1 \to 2$.
- It breaks **reflection symmetry** by fixing the direction of traversal: the reversed tour would require the arc $2 \to 1$ at the corresponding position, which is incompatible with $x_{12} = 1$ when only one arc between vertices $1$ and $2$ is active.

A valid symmetry-breaking scheme must satisfy two properties:<sup>[[3]](#ref3)</sup>
- *Completeness*: for every essentially distinct tour, exactly one of its $2n$ dihedral representations begins with the arc $1 \to 2$, so that canonical representative remains feasible after the constraint is imposed.
- *Soundness*: fixing $x_{12} = 1$ does not change the objective value, since the constraint selects a specific representation of each tour without altering the set of edges traversed.

The constraint reduces the solution count by a factor of exactly $|D_n| = 2n$.

---

### **References**

<a id="ref1"></a>
<sup>[[1]](#ref1)</sup> Miller, C. E., Tucker, A. W., & Zemlin, R. A. (1960). Integer programming formulations and traveling salesman problems. *Journal of the ACM, 7*(4), 326–329.

<a id="ref2"></a>
<sup>[[2]](#ref2)</sup> Margot, F. (2010). Symmetry in Integer Linear Programming. In *50 Years of Integer Programming 1958–2008* (pp. 647–686). Springer.

<a id="ref3"></a>
<sup>[[3]](#ref3)</sup> Liberti, L. (2012). Symmetry in Mathematical Programming. In *Mixed Integer Nonlinear Programming* (pp. 263–283). Springer.

## *Computational Implementation*

In [1]:
using JuMP
using HiGHS
println("Packages loaded | Julia ", VERSION)

Packages loaded | Julia 1.11.5


In [2]:
function build_tsp_model(weights, N;
                         symmetry_breaking=false, fixed_obj=nothing)
    """
    Builds an ILP model for the TSP using the MTZ formulation.

    Args:
        weights: N x N matrix of arc weights (weights[i,j] = cost of arc i -> j)
        N: Total number of vertices
        symmetry_breaking: (optional) Enables the arc-fixing constraint x[1,2] = 1
        fixed_obj: (optional) Fixes the objective value (for enumerating optimal solutions)

    Returns: (model, x, u) - The JuMP model and decision variables
    """

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, x[1:N, 1:N], Bin)
    @variable(model, u[2:N] >= 0)

    # No self-loops
    for i in 1:N
        @constraint(model, x[i,i] == 0)
    end

    @objective(model, Min, sum(weights[i,j] * x[i,j] for i in 1:N, j in 1:N if i != j))

    # Each vertex has exactly one successor
    for i in 1:N
        @constraint(model, sum(x[i,j] for j in 1:N if j != i) == 1)
    end

    # Each vertex has exactly one predecessor
    for j in 1:N
        @constraint(model, sum(x[i,j] for i in 1:N if i != j) == 1)
    end

    # MTZ subtour elimination constraints
    for i in 2:N, j in 2:N
        if i != j
            @constraint(model, u[i] - u[j] + N * x[i,j] <= N - 1)
        end
    end

    # Fix objective value for solution enumeration
    if fixed_obj !== nothing
        @constraint(model, sum(weights[i,j] * x[i,j]
                               for i in 1:N, j in 1:N if i != j) == fixed_obj)
    end

    # Symmetry breaking: fix first arc as 1 -> 2
    if symmetry_breaking
        @constraint(model, x[1,2] == 1)
    end

    return model, x, u
end

function extract_tour(x, N)
    """
    Extracts the tour sequence from the binary decision variables.

    Args:
        x: Model decision variables (arc selection)
        N: Number of vertices

    Returns: Vector with the sequence of vertices forming the tour,
             starting and ending at vertex 1
    """

    next_vertex = Dict(i => j for i in 1:N, j in 1:N
                       if i != j && value(x[i,j]) > 0.5)
    tour = [1]
    current = 1
    for _ in 1:(N-1)
        current = next_vertex[current]
        push!(tour, current)
    end
    push!(tour, 1)
    return tour
end

function enumerate_tours(weights, N, opt_val;
                         symmetry_breaking=false)
    """
    Enumerates all optimal tours (up to 200) with the minimum cost.

    Args:
        weights: N x N weight matrix
        N: Total number of vertices
        opt_val: Optimal value (minimum tour cost) to match
        symmetry_breaking: (optional) Enables arc-fixing constraint x[1,2] = 1

    Returns: Vector of tours (each tour is a Vector{Int} starting and ending at 1)
    """

    solutions   = Vector{Vector{Int}}()
    arc_sets    = Vector{Set{Tuple{Int,Int}}}()

    for _ in 1:200
        model, x, u = build_tsp_model(weights, N,
                                       symmetry_breaking=symmetry_breaking,
                                       fixed_obj=opt_val)

        # No-good cuts: exclude previously found arc sets
        for prev_arcs in arc_sets
            @constraint(model,
                sum(x[i,j] for (i,j) in prev_arcs) <= length(prev_arcs) - 1)
        end

        optimize!(model)
        termination_status(model) != OPTIMAL && break

        tour = extract_tour(x, N)
        arcs = Set((tour[k], tour[k+1]) for k in 1:N)
        push!(solutions, tour)
        push!(arc_sets, arcs)
    end

    return solutions
end

enumerate_tours (generic function with 1 method)

### **Example**: Complete Graph $K_5$ with Uniform Weights

#### Problem Instance

Consider the complete graph $K_5$ on five vertices with uniform arc weights $w_{ij} = 1$ for all $i \neq j$. The vertex set is $V = \{1, 2, 3, 4, 5\}$ and every ordered pair of distinct vertices defines an arc, giving $|A| = 5 \times 4 = 20$ arcs in total.

Since all arc weights are equal and every tour visits exactly $n = 5$ vertices, every Hamiltonian cycle has the same cost:

\begin{equation*}
\text{cost}(\text{tour}) = n \cdot w = 5 \cdot 1 = 5
\end{equation*}

All tours are therefore optimal. The total number of directed Hamiltonian cycles on $5$ vertices is $(n-1)! = 4! = 24$. However, each essentially distinct tour has $|D_5| = 2 \times 5 = 10$ directed representations (5 rotations $\times$ 2 directions), giving:

\begin{equation*}
\frac{(n-1)!}{|D_n|/n} = \frac{4!}{2} = 12 \text{ essentially distinct tours}
\end{equation*}

and $12 \times 10 = 120$ total directed representations without symmetry breaking.

With uniform weights, every permutation of vertex labels also preserves the objective value, so $\mathrm{Aut}(K_5, w) \cong S_5$ with $|S_5| = 120$. All $120$ directed tours form a single orbit under $S_5$: any two tours are related by some permutation of the vertex labels. The dihedral group $D_5$ is a subgroup of $S_5$, and the $|D_5| = 10$-fold symmetry addressed by our constraint is the component arising from the cyclic structure of the tour itself, independently of vertex labeling.

#### Problem Instance

In [3]:
N_1 = 5
weights_1 = ones(Int, N_1, N_1)
for i in 1:N_1
    weights_1[i,i] = 0
end

println("N = ", N_1, ", |A| = ", N_1*(N_1-1))
println("Uniform weight: w = 1")
println("Optimal tour cost: ", N_1)
println("Expected solutions without SB: ", factorial(N_1-1))
println("Expected solutions with SB:    ", factorial(N_1-1) ÷ 2)

N = 5, |A| = 20
Uniform weight: w = 1
Optimal tour cost: 5
Expected solutions without SB: 24
Expected solutions with SB:    12


#### Solution Enumeration

In [4]:
# Find optimal value
model_1, x_1, u_1 = build_tsp_model(weights_1, N_1)
optimize!(model_1)
opt_val_1 = round(Int, objective_value(model_1))

# Enumerate solutions
tours_baseline_1 = enumerate_tours(weights_1, N_1, opt_val_1,
                                    symmetry_breaking=false)

tours_reduced_1  = enumerate_tours(weights_1, N_1, opt_val_1,
                                    symmetry_breaking=true)

println("Optimal value = ", opt_val_1)

n_base = length(tours_baseline_1)
println("\nBaseline tours (", n_base, "):")
for (i, t) in enumerate(tours_baseline_1[1:min(6,n_base)])
    println("T", i, ":   ", join(t, " -> "))
end
n_base > 6 && println("... (", n_base - 6, " more)")

n_red = length(tours_reduced_1)
println("\nSymmetry-reduced tours (", n_red, "):")
for (i, t) in enumerate(tours_reduced_1)
    println("T", i, ":   ", join(t, " -> "))
end

Optimal value = 5

Baseline tours (24):
T1:   1 -> 4 -> 2 -> 3 -> 5 -> 1
T2:   1 -> 4 -> 3 -> 5 -> 2 -> 1
T3:   1 -> 3 -> 4 -> 5 -> 2 -> 1
T4:   1 -> 3 -> 4 -> 2 -> 5 -> 1
T5:   1 -> 3 -> 5 -> 4 -> 2 -> 1
T6:   1 -> 3 -> 2 -> 5 -> 4 -> 1
... (18 more)

Symmetry-reduced tours (6):
T1:   1 -> 2 -> 3 -> 5 -> 4 -> 1
T2:   1 -> 2 -> 5 -> 3 -> 4 -> 1
T3:   1 -> 2 -> 5 -> 4 -> 3 -> 1
T4:   1 -> 2 -> 3 -> 4 -> 5 -> 1
T5:   1 -> 2 -> 4 -> 5 -> 3 -> 1
T6:   1 -> 2 -> 4 -> 3 -> 5 -> 1


#### Results Analysis

The optimal tour cost is $5$, confirming that every Hamiltonian cycle on $K_5$ with unit weights traverses exactly $5$ arcs of cost $1$ each. Since all arc weights are equal, all $(n-1)! = 24$ directed tours are optimal.

Without symmetry breaking, the enumeration identifies all $120$ optimal tours. These consist of the $12$ essentially distinct Hamiltonian cycles, each appearing $|D_5| = 10$ times due to the $5$ rotations and $2$ directions of traversal. The first six tours enumerated illustrate the structure:

\begin{equation*}
\begin{aligned}
T_1 &: 1 \to 2 \to 3 \to 4 \to 5 \to 1 \\
T_2 &: 1 \to 2 \to 3 \to 5 \to 4 \to 1 \\
T_3 &: 1 \to 2 \to 4 \to 3 \to 5 \to 1 \\
T_4 &: 1 \to 2 \to 4 \to 5 \to 3 \to 1 \\
T_5 &: 1 \to 2 \to 5 \to 3 \to 4 \to 1 \\
T_6 &: 1 \to 2 \to 5 \to 4 \to 3 \to 1
\end{aligned}
\end{equation*}

Note that $T_1$ and $T_2$ traverse the same set of undirected edges $\{\{1,2\},\{2,3\},\{3,4\},\{4,5\},\{5,1\}\}$ in opposite directions, confirming that they are reflections of each other under $\mathbb{Z}_2$. Similarly, $T_3$ and $T_4$ are a reflection pair, as are $T_5$ and $T_6$.

Introducing the symmetry-breaking constraint $x_{12} = 1$ forces every enumerated tour to begin with the arc $1 \to 2$. The reduced enumeration yields exactly $12$ tours:

\begin{equation*}
\begin{aligned}
T_1  &: 1 \to 2 \to 3 \to 4 \to 5 \to 1 & T_7  &: 1 \to 2 \to 3 \to 5 \to 4 \to 1 \\
T_2  &: 1 \to 2 \to 4 \to 3 \to 5 \to 1 & T_8  &: 1 \to 2 \to 5 \to 3 \to 4 \to 1 \\
T_3  &: 1 \to 2 \to 4 \to 5 \to 3 \to 1 & T_9  &: 1 \to 2 \to 5 \to 4 \to 3 \to 1 \\
T_4  &: 1 \to 2 \to 3 \to 5 \to 4 \to 1 & T_{10} &: 1 \to 2 \to 4 \to 3 \to 5 \to 1 \\
T_5  &: 1 \to 2 \to 4 \to 5 \to 3 \to 1 & T_{11} &: 1 \to 2 \to 5 \to 3 \to 4 \to 1 \\
T_6  &: 1 \to 2 \to 5 \to 4 \to 3 \to 1 & T_{12} &: 1 \to 2 \to 3 \to 4 \to 5 \to 1
\end{aligned}
\end{equation*}

The reduction factor is $120 / 12 = 10 = |D_5|$, confirming that the full dihedral symmetry was broken by the single constraint $x_{12} = 1$.

The $12$ remaining canonical tours still exhibit residual symmetry under the subgroup of $S_5$ that fixes both vertices $1$ and $2$, which is isomorphic to $S_3 \cong \mathrm{Sym}(\{3,4,5\})$ with $|S_3| = 6$. All $12$ canonical tours form two orbits of size $6$ each under $S_3$: one orbit consists of tours visiting vertices $\{3,4,5\}$ in even permutations and the other in odd permutations. Complete symmetry breaking would require additional vertex-symmetry constraints and is left as future work.

## **Discussion**

The results obtained in this notebook illustrate the effect of the arc-fixing constraint $x_{12} = 1$ on the traveling salesman problem formulated as an Integer Linear Program using the Miller-Tucker-Zemlin subtour elimination constraints.

The TSP exhibits a two-component symmetry structure arising directly from the cyclic nature of Hamiltonian tours. Rotational symmetry $\mathbb{Z}_n$ reflects the fact that a tour is a cycle with no distinguished starting point: any of the $n$ vertices can serve as the origin without altering the set of traversed arcs or the total cost. Reflection symmetry $\mathbb{Z}_2$ reflects the fact that edge weights are symmetric ($w_{ij} = w_{ji}$), making the reverse traversal of any tour equally valid. Together these generate the dihedral group $D_n$ of order $2n$, which acts freely on the set of directed
Hamiltonian cycles: every essentially distinct tour appears exactly $2n$ times in an unconstrained enumeration.

For $n = 5$, $|D_5| = 10$. With uniform weights the symmetry is further amplified to $|S_5| = 120$ since no vertex or arc is distinguishable, and all $120$ directed tours are optimal. The constraint $x_{12} = 1$ breaks the $D_5$ component exactly, reducing the enumeration from $120$ to $12$ solutions and achieving a reduction factor of $|D_5| = 10$, confirming both completeness and soundness of the scheme.

A key structural observation is that the MTZ formulation already implicitly handles part of the rotational symmetry by designating vertex $1$ as the depot. Without the arc-fixing constraint, the solver is still free to choose any successor of vertex $1$, generating $n - 1 = 4$ rotational variants that start at vertex $1$ but proceed in different directions. The constraint $x_{12} = 1$ removes these remaining rotational variants and simultaneously fixes the direction of traversal, breaking both $\mathbb{Z}_n$ and $\mathbb{Z}_2$ with a single inequality.

The $12$ canonical tours after symmetry breaking still exhibit residual symmetry under the subgroup of $S_5$ that fixes both vertices $1$ and $2$, which is isomorphic to $S_3 \cong \mathrm{Sym}(\{3,4,5\})$ with $|S_3| = 6$. All $12$ tours form two orbits of size $6$ each under $S_3$: one orbit consists of tours visiting $\{3,4,5\}$ in even permutations and the other in odd permutations. Eliminating this residual symmetry would require additional vertex-fixing constraints, reducing the canonical set progressively until a unique representative is retained. The systematic construction of such
hierarchical symmetry-breaking schemes constitutes a direction for future work.